In [3]:
import gym
from gym import Wrapper
from stable_baselines3 import PPO
import time
import numpy as np
class VisualNoiseWrapper(Wrapper):
    def __init__(self, env, noise_level=0.1):
        super().__init__(env)
        self.noise_level = noise_level
    
    def reset(self):
        obs, info = self.env.reset()
        return self._add_noise(obs), info
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return self._add_noise(obs), reward, terminated, truncated, info
    
    def _add_noise(self, obs):
        noise = np.random.randn(*obs.shape) * self.noise_level
        return np.clip(obs + noise, 0, 255).astype(np.uint8)

env = VisualNoiseWrapper(gym.make("ALE/SpaceInvaders-v5",render_mode="human"), noise_level=25)


# 加载训练好的模型
model = PPO.load("ppo_spaceinvaders_200epochs_256bs")


# 初始化统计容器
episode_rewards = []
episode_durations = []  # 存活时间（步数）

# 测试10个episode
for episode in range(10):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    steps = 0
    
    while not done:
        # 添加批次维度并将观测输入模型
        action, _ = model.predict(obs[None, ...], deterministic=True)
        
        # 执行动作
        obs, reward, terminated, truncated, info = env.step(action[0])
        done = terminated or truncated
        
        total_reward += reward
        steps += 1
        
        # 添加微小延迟使画面可见
        time.sleep(0.01)
    
    # 计算指标
    survival_time = steps / 60  # 转换为秒（假设60FPS）
    
    # 记录数据
    episode_rewards.append(total_reward)
    episode_durations.append(survival_time)

    
    # 打印单局结果
    print(f"Episode {episode+1}:")
    print(f"  Total Reward: {total_reward}")
    print(f"  Survival Time: {survival_time:.2f}s")


# 计算综合统计数据
avg_reward = np.mean(episode_rewards)
std_reward = np.std(episode_rewards)
avg_duration = np.mean(episode_durations)


# 输出最终报告
print("\n=== Final Test Report ===")
print(f"Average Reward: {avg_reward:.1f} ± {std_reward:.1f}")
print(f"Average Survival Time: {avg_duration:.1f}s")


# 关闭环境
env.close()

D:\anaconda\envs\pytorch\lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: code() takes at most 16 arguments (18 given)
  warnings.warn(
D:\anaconda\envs\pytorch\lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() takes at most 16 arguments (18 given)
  warnings.warn(


Episode 1:
  Total Reward: 265.0
  Survival Time: 8.40s
Episode 2:
  Total Reward: 155.0
  Survival Time: 6.43s
Episode 3:
  Total Reward: 265.0
  Survival Time: 17.23s
Episode 4:
  Total Reward: 540.0
  Survival Time: 14.30s
Episode 5:
  Total Reward: 180.0
  Survival Time: 8.62s
Episode 6:
  Total Reward: 360.0
  Survival Time: 11.82s
Episode 7:
  Total Reward: 155.0
  Survival Time: 8.05s
Episode 8:
  Total Reward: 325.0
  Survival Time: 10.13s
Episode 9:
  Total Reward: 310.0
  Survival Time: 9.43s
Episode 10:
  Total Reward: 110.0
  Survival Time: 6.40s

=== Final Test Report ===
Average Reward: 266.5 ± 120.7
Average Survival Time: 10.1s
